# Engine comparison — base vs authored vs gpt551 (Colab)

Scores the three **4-bit** small models on the quarantined eval sets + the new `benchmarks/api_bench`.
These are CUDA-only (bitsandbytes 4-bit), so they must run on a GPU runtime — not the Mac.

The **frontier reference (gpt-4.1)** was already run on all sets locally; its numbers are baked into the
final cell, so the last table shows the full **4-engine** comparison once this finishes.

**Before you start:** `Runtime → Change runtime type → GPU` (T4 is fine). ~5–8 min total.

You need two adapters on Drive (each a folder with `adapter_config.json` + `adapter_model.safetensors`):
the **gpt551** adapter (the `sft-v3` folder from your training run) and the **4-bit authored** adapter.

### 1. Mount Drive (where your adapters live)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Clone the repo and check out this branch
(has `benchmarks/api_bench` + the eval code)

In [ ]:
import os
!git clone -q https://github.com/f15cubing/slm-deid.git /content/slm-deid
os.chdir('/content/slm-deid')
!git checkout -q worktree-eval-engine-comparisons
!git log --oneline -1

### 3. Install dependencies

In [ ]:
!pip install -q unsloth
!pip install -q trl datasets faker pyyaml

### 4. Point to your two adapters on Drive
**Edit these two paths** to match your Drive. The asserts fail fast if a path is wrong.

In [ ]:
AUTHORED = "/content/drive/MyDrive/slm-deid-authored/sft-v3-colab-authored"  # <-- EDIT
GPT551   = "/content/drive/MyDrive/slm-deid-gpt551/sft-v3"                   # <-- EDIT

import os
for name, p in [("AUTHORED", AUTHORED), ("GPT551", GPT551)]:
    cfg = os.path.join(p, "adapter_config.json")
    assert os.path.exists(cfg), f"{name}: no adapter_config.json at {p!r} — fix the path"
print("adapters OK")

### 5. Run the 3-way (base vs authored vs gpt551) on every set
`base` = the prompted 4-bit Qwen3 (no adapter). Each set prints its own comparison table and saves
per-model JSON reports under `outputs/eval_bench_<set>/`.

In [ ]:
SPLITS = [
    "eval/hardcases",
    "eval/adversarial",
    "eval/heldout_names",
    "eval/ood_probe",
    "benchmarks/api_bench",
]
for split in SPLITS:
    name = split.split('/')[-1]
    print(f"\n================= {name} =================")
    !python -m src.eval.run --split {split} --compare base {AUTHORED} {GPT551} --backend unsloth --report-dir outputs/eval_bench_{name}

### 6. Combined 4-engine matrix (adds the gpt-4.1 frontier reference)
Reads the reports just written and appends the frontier numbers (run earlier). Copy this output back
to complete `docs/eval-engine-comparison.md`.

In [ ]:
import glob, json

COLS = ["recall", "leakage_rate", "over_tag_rate", "integrity_violation_rate", "pass_rate", "consistency"]
HDR  = "| engine | recall | leakage | over_tag | integrity | pass | consistency |"
SEP  = "|---|---|---|---|---|---|---|"

# gpt-4.1 frontier reference (run locally, 2026-07-10). None where undefined.
FRONTIER = {
    "hardcases":     [0.778, 0.118, 0.000, 0.020, 0.882, 0.625],
    "adversarial":   [0.963, 0.025, 0.025, 0.025, 0.950, 0.857],
    "heldout_names": [1.000, 0.000, 0.081, 0.000, 0.919, 0.892],
    "ood_probe":     [1.000, 0.000, 0.000, 0.000, 1.000, 1.000],
    "api_bench":     [0.870, 0.109, 0.098, 0.054, 0.848, None],
}

def fmt(v):
    return "–" if v is None else f"{v:.3f}"

for split in SPLITS:
    name = split.split('/')[-1]
    print(f"\n### {name}\n")
    print(HDR); print(SEP)
    for fp in sorted(glob.glob(f"outputs/eval_bench_{name}/*.json")):
        d = json.load(open(fp)); o = d["overall"]
        print("| " + d["label"] + " | " + " | ".join(fmt(o.get(c)) for c in COLS) + " |")
    if name in FRONTIER:
        print("| frontier-gpt4.1 | " + " | ".join(fmt(v) for v in FRONTIER[name]) + " |")

### 7. Send results back
Copy the tables from cell 5 (or the combined matrix from cell 6) into the chat, and they'll be folded
into `docs/eval-engine-comparison.md`. Or zip the raw reports:

```python
!cd /content/slm-deid && zip -qr /content/drive/MyDrive/eval_bench_reports.zip outputs/eval_bench_*
```